# Lorehaven pricing: the analysis, step by step

This notebook walks through the whole analysis using the tables the pipeline saved (`outputs/results/`), so it
runs without the raw data. Every step is a group average or a line of arithmetic you can check by hand.

**Why averages are enough.** Inside the discounting regions, Lorehaven chose at random which prospects got an
offer (25%) and how deep it was ($1-$5, equal shares). So the people at each price are alike in everything except
price, and the difference between two group averages *is* the effect of the price difference.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from lorehaven.projection import Policy, load_rates, project
from lorehaven.recommendation import named_policies, minimum_detectable_difference

RESULTS = ROOT / "outputs" / "results"
metrics = json.loads((RESULTS / "metrics.json").read_text())
value = lambda key: metrics[key]["value"]
pd.set_option("display.precision", 3)

## 1. Did discounts work? Sign-up and stay rates at each price
Rows are the price paid; columns are the base-price era. Sign-up falls steadily as price rises, in both eras.

In [2]:
signup = pd.read_csv(RESULTS / "signup_by_price.csv")
signup.pivot(index="membership_price", columns="base_price", values="rate").style.format("{:.1%}", na_rep="")

base_price,12,15
membership_price,,
7,25.2%,
8,20.4%,
9,15.7%,
10,12.1%,17.3%
11,8.4%,13.5%
12,6.0%,10.1%
13,,7.4%
14,,5.3%
15,,3.5%


## 2. Who does a $5 offer win? ($12 era)
For each segment: the join rate at list price (would join anyway) and with $5 off. The offer's **uplift** is the
difference, and the **incremental share** is uplift ÷ join rate at $5 off: the share of $5-offer joiners who would
not have joined without it (assuming nobody is put off by an offer).

In [3]:
cells = pd.read_csv(RESULTS / "signup_by_segment_and_discount.csv")
at5 = cells[cells.discount_depth == 5].set_index("segment")
at5[["share_of_prospects", "list_rate", "rate", "uplift", "incremental_share", "n"]]

,share_of_prospects,list_rate,rate,uplift,incremental_share,n
segment,,,,,,
cold,0.647,0.005,0.129,0.124,0.961,25854
lapsed_low_podcast,0.017,0.064,0.242,0.178,0.735,715
warm_listener,0.208,0.137,0.447,0.310,0.694,8267
win_back,0.128,0.211,0.556,0.345,0.620,5200


Check one by hand: warm listeners.

In [4]:
w = at5.loc["warm_listener"]
print(f"join at list {w.list_rate:.3f}, with $5 off {w.rate:.3f}, uplift {w.rate - w.list_rate:.3f}, "
      f"incremental share {(w.rate - w.list_rate) / w.rate:.0%}")

join at list 0.137, with $5 off 0.447, uplift 0.310, incremental share 69%


## 3. Who is price-sensitive once they are members? ($12 era)
Share of members who stay, by tenure and price. New members (first renewal) react strongly to price; established
members mostly stay at any price. That is why the recommendation discounts the first two renewals and not later.

In [5]:
stay = pd.read_csv(RESULTS / "stay_by_tenure_and_price.csv")
stay.pivot_table(index=["segment", "tenure_group"], columns="membership_price", values="rate").style.format("{:.0%}")

## 4. The 2019 base cut ($15 → $12): direction, not size
Not randomized: we compare list-price sign-up in the cut regions with Panem, which never changed price.
The gap was negative in every quarter before the cut and positive in every quarter after, but it already moved by
a few points between quarters with no price change, so the size is not pinned down.

In [6]:
gap = pd.read_csv(RESULTS / "base_cut_gap_by_quarter.csv")
gap.assign(gap_points=100 * gap.gap)[["quarter_index", "cut_regions", "panem", "gap_points", "after_cut"]]

,quarter_index,cut_regions,panem,gap_points,after_cut
0,0,7.154e-03,0.014,-0.689,False
1,1,4.234e-02,0.058,-1.519,False
2,2,1.527e-02,0.027,-1.135,False
3,3,1.694e-02,0.028,-1.093,False
4,4,2.478e-02,0.042,-1.739,False
5,5,6.117e-04,0.002,-0.098,False
6,6,9.904e-02,0.125,-2.625,False
7,7,9.174e-03,0.017,-0.753,False
8,8,6.808e-02,0.093,-2.444,False
9,9,1.797e-02,0.001,1.683,True


## 5. First-quarter value of an offer policy
"Everyone gets $5 off" = for each segment, share of prospects × sign-up rate in the random $5 group × $7, summed.
(This is what inverse-propensity weighting computes; the appendix confirms it.)

In [7]:
by_hand = (at5.share_of_prospects * at5.rate * 7).sum()
print(f"by hand {by_hand:.3f}   pipeline {value('first_quarter.everyone_5_off'):.3f}   today (observed) {value('first_quarter.today'):.3f}")

by hand 1.763   pipeline 1.763   today (observed) 0.885


## 6. The projection: two lines of arithmetic per quarter
For each group of people: **new members = prospects × sign-up rate at their price**, and **members kept = members ×
stay rate for their tenure and price**. Rates are looked up in the tables above; nothing is fitted. Here is the
first quarter of the recommended policy by hand, against `project()`.

In [8]:
rates = load_rates(RESULTS / "metrics.json")
share = {"high_podcast": rates.share_high_podcast, "low_podcast": 1 - rates.share_high_podcast}
segment = {"high_podcast": "warm_listener", "low_podcast": "cold"}
first_quarter_members = sum(1000 * share[s] * rates.signup[segment[s], 5] for s in share)
result = project(named_policies()["recommended"], rates)
print(f"by hand {first_quarter_members:.1f} members   project() {result['quarters'][0]['members']:.1f}")

by hand 206.4 members   project() 206.4


## 7. Comparing policies (3 years, per 1,000 new prospects)

In [9]:
names = ["today", "offer_5_members_1_off", "offer_5_members_2_off", "offer_5_members_5_off", "recommended"]
pd.DataFrame({n: {k: value(f"policy.{n}.{k}") for k in ("revenue_per_1000_prospects", "members_at_end_per_1000",
              "average_quarterly_churn", "discount_cost_per_1000")} for n in names}).T

,revenue_per_1000_prospects,members_at_end_per_1000,average_quarterly_churn,discount_cost_per_1000
today,20227.415,259.080,0.113,6233.947
offer_5_members_1_off,39059.431,345.042,0.229,6971.789
offer_5_members_2_off,40751.172,403.387,0.180,10639.755
offer_5_members_5_off,38013.786,544.165,0.098,25799.138
recommended,44942.208,455.699,0.137,13482.387


## 8. Is a 10% holdout big enough?
Smallest difference a test catches 80% of the time at 5% significance:
2.80 × √( p₁(1−p₁)/n₁ + p₂(1−p₂)/n₂ ), where p is each arm's rate and n its size.

In [10]:
traffic = value("holdout.prospects_per_quarter")
p_list, p_offer = value("rates.by_price.signup.base12.price12"), value("rates.by_price.signup.base12.price7")
n_list, n_offer = 0.10 * traffic, 0.85 * traffic
print(f"sign-up: detectable {minimum_detectable_difference(p_list, n_list, p_offer, n_offer):.2%} vs real gap {p_offer - p_list:.1%}")
print(f"renewal after 3 quarters: detectable {value('holdout.mde_renewal_list10_q3'):.1%}")

sign-up: detectable 0.48% vs real gap 19.2%
renewal after 3 quarters: detectable 2.0%


## 9. Cross-check against the model-based appendix
The same questions answered with regression models (`python -m lorehaven.robustness`).

In [11]:
pd.read_csv(ROOT / "outputs" / "robustness" / "simple_vs_advanced.csv")[["question", "simple", "advanced", "same_direction"]]

,question,simple,advanced,same_direction
0,Sign-up odds change per $1 off ($12 era),0.396,0.461,True
1,"First-quarter revenue per prospect, today",0.885,0.885,True
2,"First-quarter revenue per prospect, $5 to ever...",1.763,1.765,True
3,Sign-up uplift at $5 off: warm_listener,0.310,0.310,True
4,Sign-up uplift at $5 off: cold,0.124,0.124,True
5,Sign-up uplift at $5 off: win_back,0.345,0.345,True
6,Sign-up uplift at $5 off: lapsed_low_podcast,0.178,0.179,True
7,2019 base cut raised list-price sign-ups (sign),0.031,0.840,True
8,"Recommended vs today, 3 years",1.222,1.325,True
9,"Recommended vs today, 10 years",0.654,0.693,True
